In [87]:
import pandas as pd
import numpy as np
import pysrt
import os
import subprocess
from pathlib import Path

In [88]:
# read signbank csv
file_path = r"..\..\..\AnimData\dictionary-export.csv"
df = pd.read_csv(file_path)

df.head()

,Signbank ID,Dataset,Lemma ID Gloss (Dutch),Lemma ID Gloss (English),Annotation ID Gloss (Dutch),Annotation ID Gloss (English),Senses (Dutch),Senses (English),Example Sentences (Dutch),Example Sentences (English),...,In the Web dictionary,Is this a proposed new sign?,Dialects,Sequential Morphology,Simultaneous Morphology,Blend Morphology,Relations to other signs,Relations to foreign signs,Tags,Notes
0,3808,NGT,#A,NaN,#A,#A,1. a...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3809,NGT,#B,NaN,#B,#B,1. b...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3810,NGT,#C,NaN,#C,#C,1. c...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3811,NGT,#D,NaN,#D,#D,1. d...,1. d...,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3812,NGT,#E,NaN,#E,#E,1. e...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [89]:
def get_handshape_from_gloss(gloss):
    row = df[df['Annotation ID Gloss (Dutch)'] == gloss]
    if not row.empty:
        return row.iloc[0]['Strong Hand']
    return None

In [90]:
def get_frames_from_timecode(timecode, fps):
    # Convert timecode to total seconds
    total_seconds = timecode.hours * 3600 + timecode.minutes * 60 + timecode.seconds + timecode.milliseconds / 1000.0
    # Calculate frame number
    frame_number = int(total_seconds * fps)
    return frame_number

In [91]:
# read srt
test = pysrt.open(r"..\..\..\AnimData\srt\M20240925_1858_260319_0_Signbank_ID_glossen.srt")

fps = 30  # frames per second

for sub in test:
    print(get_frames_from_timecode(sub.start, fps), get_frames_from_timecode(sub.end, fps), get_handshape_from_gloss(sub.text))

10 26 V
33 39 S
49 62 1
71 81 T
90 128 -


In [92]:
def get_matches_from_csv(csv_path):
    matches = []
    with open(csv_path, 'r') as f:
        next(f)  # Skip the header line
        for line in f:
            # rank, handshape_file, distance
            parts = line.strip().split(',')
            if len(parts) == 3:
                matches.append((int(parts[0]), parts[1], float(parts[2])))
    return matches

In [97]:
# go through each fbx and compare
fbx_folder = r"..\..\fbx"

exe_loc = r"..\..\build\Release\shapeseg.exe"
repo_root = Path(exe_loc).resolve().parents[2]
exe = repo_root / "build" / "Release" / "shapeseg.exe"

segment_file = r"..\..\output\handshape_segments.csv"

matches = dict()

for fbx_file in os.listdir(fbx_folder):
    if fbx_file.endswith(".fbx"):
        matches[fbx_file] = []

        fbx = repo_root / "fbx" / fbx_file

        # Update output files by running the command
        result = subprocess.run(
            [str(exe), "s", str(fbx)],
            cwd=repo_root,
            check=True,
            capture_output=True,
            text=True,
        )

        # Get number of segments
        file = open(segment_file, 'r')
        segments = [(line.split(',')[0], line.split(',')[1]) for line in file.readlines()[1:]]
        file.close()
        print(f"Processed {fbx_file}: {len(segments)} segments found.")

        # Get the matches for each segment
        for i in range(len(segments)):
            result = subprocess.run(
                [str(exe), "m", str(fbx), str(i)],
                cwd=repo_root,
                check=True,
                capture_output=True,
                text=True,
            )

            # get matches
            match_file = repo_root / "output" / "handshape_matches.csv"
            match_list = get_matches_from_csv(match_file)
            matches[fbx_file] += match_list



Processed M20241107_6263_260710_2.fbx: 4 segments found.
Processed M20250916_4405_260709_0.fbx: 1 segments found.
Processed M20260119_1078_260713_1.fbx: 11 segments found.
Processed M20260120_1291_260713_0.fbx: 9 segments found.
Processed M20260126_1564_260713_1.fbx: 11 segments found.
Processed M20260126_1578_260713_0.fbx: 8 segments found.
Processed M20260127_2139_260713_0.fbx: 7 segments found.
Processed M20260506_0579_260713_0.fbx: 11 segments found.
Processed M20260506_0594_260713_4.fbx: 16 segments found.
Processed M20260506_0604_260713_0.fbx: 16 segments found.


In [98]:
# make a dataframe from the matches dictionary
data = []

for fbx_file, match_list in matches.items():
    for match in match_list:
        data.append({
            "fbx_file": fbx_file,
            "rank": match[0],
            "handshape_file": match[1],
            "distance": match[2]
        })

matches_df = pd.DataFrame(data)

matches_df.head()

,fbx_file,rank,handshape_file,distance
0,M20241107_6263_260710_2.fbx,1,../HandformSession/HandformsTrimmedMirrored\Be...,1.59361
1,M20241107_6263_260710_2.fbx,2,../HandformSession/HandformsTrimmedMirrored\M-...,1.69983
2,M20241107_6263_260710_2.fbx,3,../HandformSession/HandformsTrimmedMirrored\P.FBX,1.84931
3,M20241107_6263_260710_2.fbx,4,../HandformSession/HandformsTrimmedMirrored\C.FBX,1.98081
4,M20241107_6263_260710_2.fbx,5,../HandformSession/HandformsTrimmedMirrored\9.FBX,2.04615
